# Evaluate the conditional Groove VAE

This notebook loads a trained checkpoint and the generated training-pair
dataset, recreates the held-out split, and runs all objective evaluation
experiments from the original notebook.


In [ ]:
# Model and evaluation-data paths
MODEL_PATH = "text2groove_model.pt"
TRAINING_PAIRS_PATH = "groove_training_pairs_genre.h5"


In [ ]:
import h5py
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import pretty_midi
import random

from sentence_transformers import SentenceTransformer
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
# Unique styles in the dataset:
# afrobeat
# afrocuban
# afrocuban/bembe
# afrocuban/calypso
# afrocuban/rhumba
# blues/shuffle
# country
# dance/breakbeat
# dance/disco
# funk
# funk/fast
# funk/latin
# funk/purdieshuffle
# funk/rock
# gospel
# highlife
# hiphop
# jazz
# jazz/fast
# jazz/funk
# jazz/fusion
# jazz/klezmer
# jazz/linear
# jazz/march
# jazz/mediumfast
# jazz/swing
# latin
# latin/bomba
# latin/bounce
# latin/brazilian
# latin/brazilian-baiao
# latin/brazilian-bossa
# latin/brazilian-frevo
# latin/brazilian-ijexa
# latin/brazilian-maracatu
# latin/brazilian-samba
# latin/brazilian-sambareggae
# latin/brazilian-songo
# latin/chacarera
# latin/dominican-merengue
# latin/merengue
# latin/reggaeton
# latin/samba
# latin/venezuelan-joropo
# latin/venezuelan-sangueo
# middleeastern
# neworleans/chacha
# neworleans/funk
# neworleans/secondline
# neworleans/shuffle
# pop
# pop/soft
# punk
# reggae
# reggae/slow
# rock
# rock/folk
# rock/halftime
# rock/indie
# rock/prog
# rock/rockabilly
# rock/shuffle
# soul
# soul/motown

GENRES = [
    "funk",
    "rock",
    "jazz",
    "hiphop",
]

FEEL_PHRASES = {
    "swung": [
        "with a swung feel",
        "with swing",
        "with delayed offbeats",
    ],
    "tight": [
        "with tight, precise timing",
        "locked to the grid",
        "with a straight pocket",
    ],
    "loose": [
        "with a relaxed pocket",
        "with laid-back timing",
        "with loose timing variation",
    ],
}

DYNAMICS_PHRASES = {
    "soft": [
        "play it softly",
        "use gentle dynamics",
        "keep the intensity low",
    ],
    "medium": [
        "use moderate dynamics",
        "keep the intensity balanced",
        "play with medium strength",
    ],
    "loud": [
        "play it loud",
        "use strong dynamics",
        "keep the intensity high",
    ],
}

PROMPT_PATTERNS = [
    "Give this a {genre} groove {feel}, {dynamics}.",
    "Humanize this as {genre} {feel}, {dynamics}.",
    "Make this sound like {genre} {feel}, {dynamics}.",
]


def _decode_h5_strings(values):
    # Decode a hdf5 string dataset to a np string array
    decoded = []
    for value in np.asarray(values):
        if isinstance(value, (bytes, np.bytes_)):
            decoded.append(value.decode("utf-8"))
        else:
            decoded.append(str(value))
    return np.asarray(decoded, dtype=str)


def create_attribute_prompt(genre, feel, overall_dynamics, random_prompt=True):
    # Build a prompt from the metadata attributes for one groove.
    # Validation/test use the first variant of each combination so repeated runs are deterministic.
    
    genre = str(genre).strip().lower() or "drum"
    feel = str(feel).strip().lower()
    overall_dynamics = str(overall_dynamics).strip().lower()

    chooser = random.choice if random_prompt else lambda values: values[0]

    return chooser(PROMPT_PATTERNS).format(
        genre=genre,
        feel=chooser(FEEL_PHRASES[feel]),
        dynamics=chooser(DYNAMICS_PHRASES[overall_dynamics]),
    )


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEXT_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"

text_encoder = SentenceTransformer(TEXT_ENCODER_NAME)
text_encoder.to(device)
text_encoder.eval()

# Freeze all text encoder parameters
for param in text_encoder.parameters():
    param.requires_grad = False

TEXT_EMB_DIM = text_encoder.get_embedding_dimension()

print("Device:", device)
print("Text embedding dim:", TEXT_EMB_DIM)

In [ ]:
# Encode the prompt strings produced by create_attribute_prompt.
# The cache avoids rerunning the frozen text encoder for wording variants
# that have already appeared in an earlier batch.
PROMPT_EMBEDDING_CACHE = {}


@torch.no_grad()
def encode_prompt_batch(prompts):
    if isinstance(prompts, str):
        prompts = [prompts]
    else:
        prompts = [str(prompt) for prompt in prompts]

    missing_prompts = list(dict.fromkeys(
        prompt
        for prompt in prompts
        if prompt not in PROMPT_EMBEDDING_CACHE
    ))

    if missing_prompts:
        missing_embeddings = text_encoder.encode(missing_prompts, batch_size=min(64, len(missing_prompts)),
            convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

        for prompt, embedding in zip(missing_prompts, missing_embeddings):
            PROMPT_EMBEDDING_CACHE[prompt] = torch.tensor(embedding, dtype=torch.float32)

    return torch.stack([PROMPT_EMBEDDING_CACHE[prompt] for prompt in prompts], dim=0).to(device)


In [ ]:
class GroovePromptDataset(Dataset):
    def __init__(self, h5_path, random_prompt=True):
        super().__init__()

        self.h5_path = h5_path
        self.random_prompt = random_prompt

        with h5py.File(h5_path, "r") as f:
            self.H = np.asarray(f["source"]["source_hits"], dtype=np.float32)
            self.O = np.asarray(f["target"]["target_offsets"], dtype=np.float32)
            self.V = np.asarray(f["target"]["target_velocities"], dtype=np.float32)

            metadata = f["metadata"]
            self.raw_styles = _decode_h5_strings(metadata["style"])
            self.feels = _decode_h5_strings(metadata["feel"])
            self.dynamics = _decode_h5_strings(metadata["overall_dynamic"])
            self.tempos = np.asarray(metadata["tempo"], dtype=np.float32)

        print("Loaded examples:", len(self.H))
        print("H shape:", self.H.shape)
        print("O shape:", self.O.shape)
        print("V shape:", self.V.shape)

    def __len__(self):
        return len(self.H)

    def __getitem__(self, idx):
        hit = self.H[idx]
        offset = self.O[idx]
        velocity = self.V[idx]
        tempo = self.tempos[idx]

        raw_style = self.raw_styles[idx]
        feel = self.feels[idx]
        overall_dynamics = self.dynamics[idx]

        prompt = create_attribute_prompt(genre=raw_style, feel=feel, overall_dynamics=overall_dynamics,
            random_prompt=self.random_prompt)

        # Loss should only apply where there is a hit.
        mask = hit.copy()

        return {
            "hits": torch.tensor(hit, dtype=torch.float32),
            "offsets": torch.tensor(offset, dtype=torch.float32),
            "velocities": torch.tensor(velocity, dtype=torch.float32),
            "mask": torch.tensor(mask, dtype=torch.float32),
            "prompt": prompt,
            "tempo": torch.tensor(tempo, dtype=torch.float32),
            "raw_style": raw_style,
            "feel": feel,
            "overall_dynamics": overall_dynamics
        }


In [ ]:
from torch.utils.data import Subset

h5_path = TRAINING_PAIRS_PATH
train_base_dataset = GroovePromptDataset(h5_path=h5_path, random_prompt=True)
print(len(train_base_dataset))
val_base_dataset = GroovePromptDataset(h5_path=h5_path, random_prompt=False)
test_base_dataset = GroovePromptDataset(h5_path=h5_path, random_prompt=False)
num_examples = len(train_base_dataset)

indices = np.arange(num_examples)
np.random.seed(42)
np.random.shuffle(indices)

val_fraction = 0.1
num_val = int(num_examples * val_fraction)
num_test = num_val * 2

val_indices = indices[:num_val]
test_indices = indices[num_val:num_test]
train_indices = indices[num_test:]

train_dataset = Subset(train_base_dataset, train_indices)
val_dataset = Subset(val_base_dataset, val_indices)
test_dataset = Subset(test_base_dataset, test_indices)

print("Train examples:", len(train_dataset))
print("Val examples:", len(val_dataset))
print("Test examples:", len(test_dataset))

In [ ]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler

train_labels = [
    train_base_dataset.raw_styles[i]
    for i in train_indices
]

train_counts = Counter(train_labels)

print("Train style counts:")
for style, count in train_counts.most_common():
    print(f"{style:15s} {count}")

alpha = 0.5

sample_weights = [
    1.0 / (train_counts[label] ** alpha)
    for label in train_labels
]

sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, 
                          shuffle=False, num_workers=0, drop_last=True)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)

batch = next(iter(train_loader))

print("hits:", batch["hits"].shape)
print("offsets:", batch["offsets"].shape)
print("velocities:", batch["velocities"].shape)
print("mask:", batch["mask"].shape)
print("prompts:", batch["prompt"][:4])
print("tempo:", batch["tempo"].shape)
print("styles:", batch["raw_style"][:8])

In [ ]:
# Text-conditioned conditional groove VAE

# transform the Sentence Transformer embedding into the size expected by the groove model
class TextConditionProjection(nn.Module):
    def __init__(self, text_emb_dim, d_model):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(text_emb_dim),
            nn.Linear(text_emb_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
        )

    def forward(self, text_emb):
        return self.net(text_emb)

class ConditionalGrooveVAE(nn.Module):
    """
    Training posterior:
        q(z | hits, target offsets, target velocities, text, tempo)

    Decoder:
        p(offsets, velocities | hits, text, tempo, z)

    At inference, z is sampled from N(0, I), so one hit pattern and prompt
    can produce multiple valid humanizations.
    """

    def __init__(self, text_emb_dim, d_model=192, latent_dim=32, n_heads=6, n_layers=3,
        dropout=0.1, n_steps=32, n_voices=9):
        super().__init__()

        self.n_steps = n_steps
        self.n_voices = n_voices
        self.n_tokens = n_steps * n_voices
        self.d_model = d_model
        self.latent_dim = latent_dim

        # Posterior encoder sees hit, target offset, and target velocity.
        self.performance_proj = nn.Linear(3, d_model)

        # Decoder sees only the fixed hit pattern.
        self.hit_proj = nn.Linear(1, d_model)

        self.step_emb = nn.Embedding(n_steps, d_model)
        self.voice_emb = nn.Embedding(n_voices, d_model)
        self.metric_emb = nn.Embedding(16, d_model)

        self.text_proj = TextConditionProjection(text_emb_dim=text_emb_dim, d_model=d_model)

        self.tempo_proj = nn.Sequential(
            nn.Linear(1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
        )

        # Add the condition to every encoder token instead of relying on one
        # condition token that the Transformer can easily ignore.
        self.encoder_condition_fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        self.posterior_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.posterior_token = nn.Parameter(torch.zeros(1, 1, d_model))

        self.mu_head = nn.Linear(d_model, latent_dim)
        self.logvar_head = nn.Linear(d_model, latent_dim)

        self.latent_proj = nn.Sequential(
            nn.Linear(latent_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        # Each decoder token receives hit content, text/tempo condition, and z.
        self.decoder_condition_fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        # Non-autoregressive Transformer decoder where all 288 output
        # locations are predicted together because the hit score is fixed.
        self.performance_decoder = nn.TransformerEncoder(decoder_layer, num_layers=n_layers)

        self.offset_head = nn.Linear(d_model, 1)
        self.velocity_head = nn.Linear(d_model, 1)

        step_ids = []
        voice_ids = []
        metric_ids = []

        for step in range(n_steps):
            for voice in range(n_voices):
                step_ids.append(step)
                voice_ids.append(voice)
                metric_ids.append(step % 16)

        self.register_buffer(
            "step_ids",
            torch.tensor(step_ids, dtype=torch.long),
        )
        self.register_buffer(
            "voice_ids",
            torch.tensor(voice_ids, dtype=torch.long),
        )
        self.register_buffer(
            "metric_ids",
            torch.tensor(metric_ids, dtype=torch.long),
        )

        nn.init.normal_(self.posterior_token, std=0.02)

    def positional_tokens(self):
        return (
            self.step_emb(self.step_ids)[None, :, :]
            + self.voice_emb(self.voice_ids)[None, :, :]
            + self.metric_emb(self.metric_ids)[None, :, :]
        )

    def get_condition(self, prompt_emb, tempo):
        batch_size = prompt_emb.shape[0]

        text_condition = self.text_proj(prompt_emb)

        tempo_norm = (
            (tempo.view(batch_size, 1) - 120.0) / 40.0
        ).clamp(-3.0, 3.0)

        tempo_condition = self.tempo_proj(tempo_norm)

        return text_condition + tempo_condition

    def encode(
        self,
        hits,
        target_offsets,
        target_velocities,
        prompt_emb,
        tempo,
    ):
        batch_size = hits.shape[0]

        # Empty positions carry no target performance information.
        masked_offsets = target_offsets * hits
        masked_velocities = target_velocities * hits

        performance_features = torch.stack([hits, masked_offsets, masked_velocities],dim=-1).reshape(batch_size, self.n_tokens, 3)

        tokens = self.performance_proj(performance_features)
        tokens = tokens + self.positional_tokens()

        condition = self.get_condition(prompt_emb, tempo)
        condition_tokens = condition[:, None, :].expand(-1, self.n_tokens, -1)

        tokens = self.encoder_condition_fusion(torch.cat([tokens, condition_tokens], dim=-1))
        posterior_token = (self.posterior_token.expand(batch_size, -1, -1) + condition[:, None, :])
        encoded = self.posterior_encoder(torch.cat([posterior_token, tokens], dim=1))

        posterior_state = encoded[:, 0, :]

        mu = self.mu_head(posterior_state)
        logvar = self.logvar_head(posterior_state).clamp(-8.0, 8.0)

        return mu, logvar

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, hits, prompt_emb, tempo, z):
        batch_size = hits.shape[0]

        hit_features = hits.reshape(batch_size, self.n_tokens, 1)

        tokens = self.hit_proj(hit_features)
        tokens = tokens + self.positional_tokens()

        condition = self.get_condition(prompt_emb, tempo)
        latent_condition = self.latent_proj(z)

        condition_tokens = condition[:, None, :].expand(-1, self.n_tokens, -1)
        latent_tokens = latent_condition[:, None, :].expand(-1, self.n_tokens, -1)

        tokens = self.decoder_condition_fusion(
            torch.cat(
                [tokens, condition_tokens, latent_tokens],
                dim=-1,
            )
        )

        decoded = self.performance_decoder(tokens)

        offsets = 0.5 * torch.tanh(
            self.offset_head(decoded)
        ).reshape(batch_size, self.n_steps, self.n_voices)

        velocities = torch.sigmoid(
            self.velocity_head(decoded)
        ).reshape(batch_size, self.n_steps, self.n_voices)

        offsets = offsets * hits
        velocities = velocities * hits

        return offsets, velocities

    def forward(self, hits, target_offsets, target_velocities, prompt_emb, tempo, sample=True):
        mu, logvar = self.encode( hits=hits, target_offsets=target_offsets, target_velocities=target_velocities,
            prompt_emb=prompt_emb, tempo=tempo)

        z = (
            self.reparameterize(mu, logvar)
            if sample
            else mu
        )

        pred_offsets, pred_velocities = self.decode(hits=hits, prompt_emb=prompt_emb, tempo=tempo, z=z)

        return pred_offsets, pred_velocities, mu, logvar

In [ ]:
# Masked conditional-VAE objective

def masked_huber_loss(pred, target, mask):
    element_loss = F.smooth_l1_loss(pred, target, reduction="none")
    element_loss = element_loss * mask
    return element_loss.sum() / mask.sum().clamp_min(1.0)


def kl_divergence(mu, logvar):
    # Mean over the batch and latent dimensions keeps the KL scale stable
    # when latent_dim changes.
    kl_per_dimension = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
    return kl_per_dimension.mean()


def vae_groove_loss(pred_offsets, pred_velocities, target_offsets, target_velocities, mask, mu,
    logvar, beta, velocity_weight=0.5):
    
    offset_loss = masked_huber_loss(pred_offsets, target_offsets, mask)
    velocity_loss = masked_huber_loss(pred_velocities, target_velocities, mask)
    reconstruction_loss = (offset_loss + velocity_weight * velocity_loss)
    kl_loss = kl_divergence(mu, logvar)
    total_loss = reconstruction_loss + beta * kl_loss

    return {
        "loss": total_loss,
        "reconstruction_loss": reconstruction_loss,
        "offset_loss": offset_loss,
        "velocity_loss": velocity_loss,
        "kl_loss": kl_loss,
    }

In [ ]:
# Initialize conditional VAE

LATENT_DIM = 16

model = ConditionalGrooveVAE(
    text_emb_dim=TEXT_EMB_DIM,
    d_model=192,
    latent_dim=LATENT_DIM,
    n_heads=6,
    n_layers=3,
    dropout=0.1,
    n_steps=32,
    n_voices=9,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4,
)

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {num_params:,}")
print(f"Latent dimensions: {LATENT_DIM}")

In [ ]:
# Train and evaluation functions

from collections import defaultdict

def move_batch_to_device(batch, device):
    hits = batch["hits"].to(device)
    offsets = batch["offsets"].to(device)
    velocities = batch["velocities"].to(device)
    mask = batch["mask"].to(device)
    tempo = batch["tempo"].to(device)

    # batch["prompt"] contains the strings generated in cell 2.
    prompt_emb = encode_prompt_batch(batch["prompt"])

    return hits, offsets, velocities, mask, prompt_emb, tempo


def train_one_epoch_vae(model, loader, optimizer, device, beta):
    model.train()

    metric_sums = defaultdict(float)

    for batch in loader:
        hits, offsets, velocities, mask, prompt_emb, tempo = move_batch_to_device(batch, device)
        pred_offsets, pred_velocities, mu, logvar = model(
            hits=hits,
            target_offsets=offsets,
            target_velocities=velocities,
            prompt_emb=prompt_emb,
            tempo=tempo,
            sample=True,
        )

        losses = vae_groove_loss(
            pred_offsets=pred_offsets,
            pred_velocities=pred_velocities,
            target_offsets=offsets,
            target_velocities=velocities,
            mask=mask,
            mu=mu,
            logvar=logvar,
            beta=beta,
            velocity_weight=1,
        )

        optimizer.zero_grad(set_to_none=True)
        losses["loss"].backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        for name, value in losses.items():
            metric_sums[name] += value.item()

    return {
        name: total / len(loader)
        for name, total in metric_sums.items()
    }


@torch.inference_mode()
def evaluate_vae(model, loader, device, beta):
    model.eval()

    metric_sums = defaultdict(float)
    style_reconstruction_sum = defaultdict(float)
    style_count = defaultdict(int)

    for batch in loader:
        hits, offsets, velocities, mask, prompt_emb, tempo = move_batch_to_device(batch, device)

        # Use posterior mean during validation to remove sampling noise.
        pred_offsets, pred_velocities, mu, logvar = model(
            hits=hits,
            target_offsets=offsets,
            target_velocities=velocities,
            prompt_emb=prompt_emb,
            tempo=tempo,
            sample=False,
        )

        losses = vae_groove_loss(
            pred_offsets=pred_offsets,
            pred_velocities=pred_velocities,
            target_offsets=offsets,
            target_velocities=velocities,
            mask=mask,
            mu=mu,
            logvar=logvar,
            beta=beta,
            velocity_weight=1,
        )

        for name, value in losses.items():
            metric_sums[name] += value.item()

        element_offset = F.smooth_l1_loss(
            pred_offsets,
            offsets,
            reduction="none",
        )
        element_velocity = F.smooth_l1_loss(
            pred_velocities,
            velocities,
            reduction="none",
        )

        element_reconstruction = (element_offset + 1 * element_velocity) * mask
        per_example_reconstruction = (element_reconstruction.sum(dim=(1, 2)) / mask.sum(dim=(1, 2)).clamp_min(1.0))

        for style, example_loss in zip(batch["raw_style"], per_example_reconstruction.cpu().tolist()):
            style_reconstruction_sum[style] += example_loss
            style_count[style] += 1

    metrics = {
        name: total / len(loader)
        for name, total in metric_sums.items()
    }

    metrics["style_losses"] = {
        style: style_reconstruction_sum[style] / style_count[style]
        for style in style_reconstruction_sum
    }

    return metrics

In [ ]:
# Helper functions for extracting groove features
# measure groove features of a pattern according to the methods described in https://doi.org/10.31751/1224

WINDOW_SIZE = 2 # 2 bars per window
HOP_SIZE = 1 # 1 bar hop size
SUBDIVISION_PER_BEAT = 4
BEATS_PER_BAR = 4

STEPS_PER_BAR = BEATS_PER_BAR * SUBDIVISION_PER_BEAT
WINDOW_STEPS = WINDOW_SIZE * STEPS_PER_BAR
HOP_STEP = HOP_SIZE * STEPS_PER_BAR

def measure_groove_features(pattern_H, pattern_O, pattern_V, center_offsets=True):
    # consolidate the matrices to 1D arrays 
    H_flat = np.array(32 * [0])
    O_flat = np.array(32 * [0.0])
    V_flat = np.array(32 * [0.0])
    for step in range(WINDOW_STEPS):
        for drum_id in range(9):
            if pattern_H[step, drum_id] == 1:
                H_flat[step] = 1
                O_flat[step] = pattern_O[step, drum_id]
                V_flat[step] = pattern_V[step, drum_id]
                break  # only consider the first hit in the priority order of drum voices 
    # print(H_flat)
    # print(O_flat)
    # print(V_flat)

    even_hits = H_flat[::2].sum()
    odd_hits = H_flat[1::2].sum()

    # determine if the pattern is in 8th notes or 16th notes
    subdivision = 8 if odd_hits / (even_hits + odd_hits) < 0.6 else 16
        
    # compute mean swing ratio
    total_duration1 = 0.0
    total_duration2 = 0.0
    number_of_off_beat_hits = 0
    if subdivision == 8:
        first_off_beat_step = 2
        increment = 4
    else:
        first_off_beat_step = 1
        increment = 2
    large_8th_note_swing = False
    large_8th_note_swing_count = 0
    for step in range(first_off_beat_step, WINDOW_STEPS, increment):
        large_8th_note_swing_local = False
        if subdivision == 8 and H_flat[step] == 0 and H_flat[step + 1] == 1: # also check next step if 8th note swing since the swung note might be there instead
            step = step + 1
            large_8th_note_swing_local = True
            large_8th_note_swing_count += 1
        if H_flat[step] == 1:
            if large_8th_note_swing_local:
                prev_step = step - 3
                next_step = step + 1
            else:
                prev_step = step - first_off_beat_step
                next_step = step + first_off_beat_step
            duration1 = (step + O_flat[step]) - (prev_step + O_flat[prev_step])
            if next_step >= WINDOW_STEPS:
                duration2 = next_step - (step + O_flat[step])
            else:
                duration2 = (next_step + O_flat[next_step]) - (step + O_flat[step])
            if duration1 > 0 and duration2 > 0:
                total_duration1 += duration1
                total_duration2 += duration2
                number_of_off_beat_hits += 1

    mean_swing_ratio = (
        total_duration1 / total_duration2
        if total_duration2 > 0
        else 0
    )

    if large_8th_note_swing_count >= 4:
        large_8th_note_swing = True   

    # center offsets 
    if center_offsets:
        mean_offset = np.mean(pattern_O[pattern_H == 1])
        pattern_O[pattern_H == 1] -= mean_offset

    #calculate std of offsets
    std_offset = np.std(pattern_O[pattern_H == 1])

    # calculate mean velocity
    mean_velocity = np.mean(pattern_V[pattern_H == 1])

    # velocity variation 
    std_velocity = np.std(pattern_V[pattern_H == 1])

    return mean_swing_ratio, subdivision, large_8th_note_swing, mean_offset, mean_velocity, std_offset, std_velocity

In [ ]:
# Conditional-VAE prompt and diversity tests

best_model_path = MODEL_PATH

checkpoint = torch.load(best_model_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


@torch.no_grad()
def encode_prompt(prompt_text):
    return encode_prompt_batch([prompt_text])


@torch.no_grad()
def predict_with_prompt_vae(model, hit_matrix, tempo_value, prompt_text, z=None, temperature=1.0):

    model.eval()

    hits = torch.as_tensor(hit_matrix, dtype=torch.float32, device=device).unsqueeze(0)
    tempo = torch.tensor([tempo_value], dtype=torch.float32, device=device,)
    prompt_emb = encode_prompt(prompt_text)

    if z is None:
        z = torch.randn(1, model.latent_dim, device=device) * temperature
    else:
        z = torch.as_tensor(z, dtype=torch.float32, device=device)

        if z.ndim == 1:
            z = z.unsqueeze(0)

    pred_offsets, pred_velocities = model.decode(hits=hits, prompt_emb=prompt_emb, tempo=tempo, z=z)

    return (pred_offsets[0].cpu().numpy(), pred_velocities[0].cpu().numpy(), z.detach().cpu().numpy())


# Grab one validation example.
example = val_base_dataset[val_indices[2]]

hit_np = example["hits"].numpy()
target_offset_np = example["offsets"].numpy()
target_velocity_np = example["velocities"].numpy()
tempo_value = float(example["tempo"])

test_prompts = []

for pattern in PROMPT_PATTERNS:
    for genre in GENRES:
        for feel_phrases in FEEL_PHRASES.values():
            for feel in feel_phrases:
                for dynamics_phrases in DYNAMICS_PHRASES.values():
                    for dynamics in dynamics_phrases:
                        prompt = pattern.format(
                            genre=genre,
                            feel=feel,
                            dynamics=dynamics,
                        )
                        test_prompts.append(prompt)

print(f"Generated {len(test_prompts)} prompts")

# test_prompts = [
#     create_attribute_prompt("jazz", "swung", "soft", random_prompt=False),
#     create_attribute_prompt("rock", "swung", "loud", random_prompt=False),
#     create_attribute_prompt("funk", "loose", "soft", random_prompt=False),
#     create_attribute_prompt("funk", "tight", "loud", random_prompt=False),
# ]

# A shared latent sample makes this a controlled prompt comparison.
torch.manual_seed(123)
shared_z = torch.randn(
    1,
    model.latent_dim,
    device=device,
)

print("PROMPT CONTROL TEST - shared z")
for prompt in test_prompts:
    pred_offset_np, pred_velocity_np, _ = predict_with_prompt_vae(
        model=model,
        hit_matrix=hit_np,
        tempo_value=tempo_value,
        prompt_text=prompt,
        z=shared_z,
    )

    active = hit_np > 0.5


In [ ]:
# Quantitative evaluation set up

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader

# Make sure the best model is loaded.
best_model_path = MODEL_PATH

checkpoint = torch.load(
    best_model_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

# Held-out test loader
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Test examples:", len(test_dataset))
print("Device:", device)

In [ ]:
# ============================================================
# EXPERIMENT 1 — TEST RECONSTRUCTION
# ============================================================

@torch.inference_mode()
def evaluate_test_reconstruction(model, loader, device):
    model.eval()

    total_hits = 0.0

    # Huber losses
    offset_huber_sum = 0.0
    velocity_huber_sum = 0.0

    # MAE / RMSE
    offset_abs_sum = 0.0
    offset_sq_sum = 0.0
    velocity_abs_sum = 0.0
    velocity_sq_sum = 0.0

    # Interpretable units
    timing_abs_ms_sum = 0.0
    velocity_abs_midi_sum = 0.0

    # For correlation
    true_offsets = []
    pred_offsets_all = []
    true_velocities = []
    pred_velocities_all = []

    for batch in loader:
        hits, offsets, velocities, mask, prompt_emb, tempo = move_batch_to_device(batch, device)

        # ----------------------------------------------------
        # Posterior reconstruction.
        #
        # sample=False means z = mu rather than stochastic
        # posterior sampling.
        # ----------------------------------------------------
        pred_offsets, pred_velocities, mu, logvar = model(
            hits=hits,
            target_offsets=offsets,
            target_velocities=velocities,
            prompt_emb=prompt_emb,
            tempo=tempo,
            sample=False,
        )

        hit_count = mask.sum().item()

        if hit_count == 0:
            continue

        total_hits += hit_count

        # ----------------------------------------------------
        # Huber loss — same type of loss used during training
        # ----------------------------------------------------
        offset_huber = F.smooth_l1_loss(pred_offsets, offsets, reduction="none")
        velocity_huber = F.smooth_l1_loss(pred_velocities, velocities, reduction="none")

        offset_huber_sum += (offset_huber * mask).sum().item()
        velocity_huber_sum += (velocity_huber * mask).sum().item()

        # ----------------------------------------------------
        # Raw prediction errors
        # ----------------------------------------------------
        offset_error = pred_offsets - offsets
        velocity_error = pred_velocities - velocities

        offset_abs_sum += (offset_error.abs() * mask).sum().item()
        offset_sq_sum += (offset_error.pow(2) * mask).sum().item()
        velocity_abs_sum += (velocity_error.abs() * mask).sum().item()
        velocity_sq_sum += (velocity_error.pow(2) * mask).sum().item()

        # ----------------------------------------------------
        # Convert normalized offset error to milliseconds
        #
        # Your MIDI reconstruction uses:
        #
        # step_duration = 60 / tempo / 4
        #
        # and offsets are fractions of this duration.
        # ----------------------------------------------------
        step_duration_ms = (60.0 / tempo / SUBDIVISION_PER_BEAT * 1000.0).view(-1, 1, 1)

        timing_abs_ms_sum += (
            offset_error.abs() * mask * step_duration_ms
        ).sum().item()

        # ----------------------------------------------------
        # Convert normalized velocity to MIDI scale 0–127
        # ----------------------------------------------------
        velocity_abs_midi_sum += (
            velocity_error.abs() * mask * 127.0
        ).sum().item()

        # ----------------------------------------------------
        # Store active hits for correlation
        # ----------------------------------------------------
        active = mask.bool()

        true_offsets.append(offsets[active].detach().cpu().numpy())
        pred_offsets_all.append(pred_offsets[active].detach().cpu().numpy())
        true_velocities.append(velocities[active].detach().cpu().numpy())
        pred_velocities_all.append(pred_velocities[active].detach().cpu().numpy())

    # --------------------------------------------------------
    # Final metrics
    # --------------------------------------------------------
    offset_huber_loss = offset_huber_sum / total_hits
    velocity_huber_loss = velocity_huber_sum / total_hits
    reconstruction_loss = offset_huber_loss + velocity_huber_loss

    offset_mae = offset_abs_sum / total_hits
    offset_rmse = np.sqrt(offset_sq_sum / total_hits)

    velocity_mae = velocity_abs_sum / total_hits
    velocity_rmse = np.sqrt(velocity_sq_sum / total_hits)

    timing_mae_ms = timing_abs_ms_sum / total_hits
    velocity_mae_midi = velocity_abs_midi_sum / total_hits

    true_offsets = np.concatenate(true_offsets)
    pred_offsets_all = np.concatenate(pred_offsets_all)
    true_velocities = np.concatenate(true_velocities)
    pred_velocities_all = np.concatenate(pred_velocities_all)

    offset_correlation = np.corrcoef(true_offsets, pred_offsets_all)[0, 1]
    velocity_correlation = np.corrcoef(true_velocities, pred_velocities_all)[0, 1]

    results = {
        "Offset Huber loss": offset_huber_loss,
        "Velocity Huber loss": velocity_huber_loss,
        "Reconstruction loss": reconstruction_loss,
        "Offset MAE": offset_mae,
        "Offset RMSE": offset_rmse,
        "Timing MAE (ms)": timing_mae_ms,
        "Velocity MAE": velocity_mae,
        "Velocity RMSE": velocity_rmse,
        "Velocity MAE (MIDI units)": velocity_mae_midi,
        "Offset correlation": offset_correlation,
        "Velocity correlation": velocity_correlation,
    }

    return results


# ------------------------------------------------------------
# Run Experiment 1
# ------------------------------------------------------------

experiment1_results = evaluate_test_reconstruction(
    model=model, loader=test_loader, device=device
)

experiment1_table = pd.DataFrame({
    "Metric": experiment1_results.keys(),
    "Value": experiment1_results.values(),
})

print("\nEXPERIMENT 1 — TEST RECONSTRUCTION")
print("=" * 60)

display(experiment1_table.round(6))

In [ ]:
# ============================================================
# EXPERIMENT 2 - DYNAMICS PROMPT CONTROL
#
# Thresholds:
#   soft   : mean velocity < 0.4832396
#   medium : 0.4832396 <= mean velocity <= 0.6237068
#   loud   : mean velocity > 0.6237068
# ============================================================

SOFT_THRESHOLD = 0.4832396
LOUD_THRESHOLD = 0.6237068


def classify_dynamics(mean_velocity):
    """
    Convert generated mean velocity into a dynamics class.
    """
    if mean_velocity < SOFT_THRESHOLD:
        return "soft"
    elif mean_velocity > LOUD_THRESHOLD:
        return "loud"
    else:
        return "medium"


@torch.inference_mode()
def evaluate_dynamics_control(model, dataset, device, n_examples=500, seed=123):
    model.eval()

    rng = np.random.default_rng(seed)
    n_examples = min(n_examples, len(dataset))
    selected_indices = rng.choice(len(dataset), size=n_examples, replace=False)

    # Reproducible latent sampling
    torch.manual_seed(seed)

    records = []

    # --------------------------------------------------------
    # For every score, generate:
    #
    #   soft
    #   medium
    #   loud
    #
    # SAME:
    #   hit pattern
    #   genre
    #   feel
    #   tempo
    #   latent z
    #
    # ONLY dynamics prompt changes.
    # --------------------------------------------------------

    for idx in selected_indices:
        example = dataset[int(idx)]

        hit_np = example["hits"].numpy()
        active = hit_np > 0.5

        if active.sum() == 0:
            continue

        tempo_value = float(example["tempo"])
        genre = str(example["raw_style"])
        feel = str(example["feel"])

        # ----------------------------------------------------
        # One shared z for all dynamics conditions
        # ----------------------------------------------------
        shared_z = torch.randn(1, model.latent_dim, device=device)

        for requested_dynamics in ["soft", "medium", "loud"]:
            prompt = create_attribute_prompt(
                genre=genre,
                feel=feel,
                overall_dynamics=requested_dynamics,
                random_prompt=False,
            )

            pred_offset_np, pred_velocity_np, _ = predict_with_prompt_vae(
                model=model,
                hit_matrix=hit_np,
                tempo_value=tempo_value,
                prompt_text=prompt,
                z=shared_z,
            )

            # ------------------------------------------------
            # Mean velocity only over active drum hits
            # ------------------------------------------------
            mean_velocity = float(pred_velocity_np[active].mean())
            predicted_dynamics = classify_dynamics(mean_velocity)
            correct = predicted_dynamics == requested_dynamics

            records.append({
                "index": int(idx),
                "genre": genre,
                "feel": feel,
                "requested_dynamics": requested_dynamics,
                "mean_velocity": mean_velocity,
                "predicted_dynamics": predicted_dynamics,
                "correct": correct,
            })

    df = pd.DataFrame(records)

    # --------------------------------------------------------
    # Overall accuracy
    # --------------------------------------------------------
    overall_accuracy = df["correct"].mean()

    # --------------------------------------------------------
    # Accuracy per requested dynamics class
    # --------------------------------------------------------
    class_accuracy = df.groupby("requested_dynamics")["correct"].mean()

    # --------------------------------------------------------
    # Mean generated velocity for each requested class
    # --------------------------------------------------------
    mean_velocity_by_prompt = df.groupby(
        "requested_dynamics"
    )["mean_velocity"].mean()

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------
    confusion = pd.crosstab(
        df["requested_dynamics"],
        df["predicted_dynamics"],
        rownames=["Requested"],
        colnames=["Predicted"],
        normalize=False,
    )

    confusion_percent = pd.crosstab(
        df["requested_dynamics"],
        df["predicted_dynamics"],
        rownames=["Requested"],
        colnames=["Predicted"],
        normalize="index",
    ) * 100

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------
    summary = {
        "N generated examples": len(df),
        "Overall dynamics accuracy": overall_accuracy,
        "Soft accuracy": class_accuracy.get("soft", np.nan),
        "Medium accuracy": class_accuracy.get("medium", np.nan),
        "Loud accuracy": class_accuracy.get("loud", np.nan),
        "Mean velocity - soft prompt": mean_velocity_by_prompt.get("soft", np.nan),
        "Mean velocity - medium prompt": mean_velocity_by_prompt.get("medium", np.nan),
        "Mean velocity - loud prompt": mean_velocity_by_prompt.get("loud", np.nan),
    }

    return df, summary, confusion, confusion_percent


# ============================================================
# RUN EXPERIMENT 2
# ============================================================

dynamics_df, experiment2_results, dynamics_confusion, dynamics_confusion_percent = (
    evaluate_dynamics_control(
        model=model,
        dataset=test_dataset,
        device=device,
        n_examples=15000,
        seed=123,
    )
)

experiment2_table = pd.DataFrame({
    "Metric": experiment2_results.keys(),
    "Value": experiment2_results.values(),
})

print("\nEXPERIMENT 2 - DYNAMICS CONTROL")
print("=" * 65)
display(experiment2_table.round(5))

print("\nDYNAMICS CONFUSION MATRIX - COUNTS")
display(dynamics_confusion)

print("\nDYNAMICS CONFUSION MATRIX - ROW %")
display(dynamics_confusion_percent.round(2))

In [ ]:
# ============================================================
# EXPERIMENT 3 - FEEL PROMPT CONTROL
#
# Genre-dependent classification thresholds:
#
# FUNK / HIPHOP:
#   swung if MEAN_SWING_RATIO >= 1.2
#
# ROCK / JAZZ:
#   swung if MEAN_SWING_RATIO >= 1.5
#
# tight:
#   MEAN_SWING_RATIO < genre-specific swing threshold
#   AND std_offset <= 0.1
#
# loose:
#   MEAN_SWING_RATIO < genre-specific swing threshold
#   AND std_offset > 0.1
#
# Swing is given priority.
# ============================================================

SWING_RATIO_THRESHOLD = 1.5
FUNK_HIPHOP_SWING_RATIO_THRESHOLD = 1.2
TIGHT_LOOSE_THRESHOLD = 0.1


# ============================================================
# GET GENRE-SPECIFIC SWING THRESHOLD
# ============================================================

def get_swing_threshold(genre):
    """
    Funk and hiphop use a lower swing threshold of 1.2.
    Rock and jazz use the default threshold of 1.5.
    """
    genre = str(genre).lower().strip()

    if genre in ["funk", "hiphop"]:
        return FUNK_HIPHOP_SWING_RATIO_THRESHOLD

    return SWING_RATIO_THRESHOLD


# ============================================================
# FEEL CLASSIFIER
# ============================================================

def classify_feel(mean_swing_ratio, std_offset, genre):
    """
    Classify generated performance using genre-dependent
    groove thresholds.

    Funk / hiphop:
        swing threshold = 1.2

    Rock / jazz:
        swing threshold = 1.5

    Priority:
        1. swung
        2. tight
        3. loose
    """
    swing_threshold = get_swing_threshold(genre)

    if mean_swing_ratio >= swing_threshold:
        return "swung"
    elif std_offset <= TIGHT_LOOSE_THRESHOLD:
        return "tight"
    else:
        return "loose"


# ============================================================
# EVALUATION FUNCTION
# ============================================================

@torch.inference_mode()
def evaluate_feel_control(model, dataset, device, n_examples=15000, seed=456):
    model.eval()

    rng = np.random.default_rng(seed)

    # If n_examples is larger than the test set,
    # use the entire test set.
    n_examples = min(n_examples, len(dataset))
    selected_indices = rng.choice(len(dataset), size=n_examples, replace=False)

    torch.manual_seed(seed)
    records = []

    # --------------------------------------------------------
    # For each test score generate:
    #
    #   tight
    #   loose
    #   swung
    #
    # SAME:
    #   hit pattern
    #   genre
    #   dynamics
    #   tempo
    #   latent z
    #
    # ONLY feel prompt changes.
    # --------------------------------------------------------

    for idx in selected_indices:
        example = dataset[int(idx)]

        hit_np = example["hits"].numpy()
        active = hit_np > 0.5

        # Need at least 2 active hits
        if active.sum() < 2:
            continue

        tempo_value = float(example["tempo"])
        genre = str(example["raw_style"]).lower().strip()

        # Get genre-specific threshold
        swing_threshold = get_swing_threshold(genre)

        # Fix dynamics so it cannot confound
        # the feel evaluation.
        dynamics = "medium"

        # ----------------------------------------------------
        # ONE shared latent vector for all three feels.
        #
        # This ensures that only the text feel condition
        # changes between tight / loose / swung.
        # ----------------------------------------------------
        shared_z = torch.randn(1, model.latent_dim, device=device)

        for requested_feel in ["tight", "loose", "swung"]:
            prompt = create_attribute_prompt(
                genre=genre,
                feel=requested_feel,
                overall_dynamics=dynamics,
                random_prompt=False,
            )

            pred_offset_np, pred_velocity_np, _ = predict_with_prompt_vae(
                model=model,
                hit_matrix=hit_np,
                tempo_value=tempo_value,
                prompt_text=prompt,
                z=shared_z,
            )

            # ------------------------------------------------
            # Extract groove features
            #
            # .copy() is important because
            # measure_groove_features() modifies offsets.
            # ------------------------------------------------
            (
                mean_swing_ratio,
                subdivision,
                large_8th_note_swing,
                mean_offset,
                mean_velocity,
                std_offset,
                std_velocity,
            ) = measure_groove_features(
                hit_np.copy(),
                pred_offset_np.copy(),
                pred_velocity_np.copy(),
            )

            # ------------------------------------------------
            # Threshold-based predicted feel
            # ------------------------------------------------
            predicted_feel = classify_feel(
                mean_swing_ratio=mean_swing_ratio,
                std_offset=std_offset,
                genre=genre,
            )

            correct = predicted_feel == requested_feel

            records.append({
                "index": int(idx),
                "genre": genre,
                "requested_feel": requested_feel,
                "swing_threshold": float(swing_threshold),
                "mean_swing_ratio": float(mean_swing_ratio),
                "std_offset": float(std_offset),
                "predicted_feel": predicted_feel,
                "correct": correct,
            })

    # ========================================================
    # DATAFRAME
    # ========================================================
    df = pd.DataFrame(records)

    # --------------------------------------------------------
    # Overall classification accuracy
    # --------------------------------------------------------
    overall_accuracy = df["correct"].mean()

    # --------------------------------------------------------
    # Classification accuracy by requested feel
    # --------------------------------------------------------
    class_accuracy = df.groupby("requested_feel")["correct"].mean()

    # ========================================================
    # FEATURE STATISTICS BY REQUESTED FEEL
    # ========================================================
    feature_means = df.groupby("requested_feel")[
        ["mean_swing_ratio", "std_offset"]
    ].mean()

    feature_stds = df.groupby("requested_feel")[
        ["mean_swing_ratio", "std_offset"]
    ].std()

    # ========================================================
    # FEATURE STATISTICS BY GENRE × REQUESTED FEEL
    # ========================================================
    genre_feature_means = df.groupby(
        ["genre", "requested_feel"]
    )[["mean_swing_ratio", "std_offset"]].mean()

    # ========================================================
    # CONFUSION MATRICES
    # ========================================================
    confusion = pd.crosstab(
        df["requested_feel"],
        df["predicted_feel"],
        rownames=["Requested"],
        colnames=["Predicted"],
        normalize=False,
    )

    confusion_percent = pd.crosstab(
        df["requested_feel"],
        df["predicted_feel"],
        rownames=["Requested"],
        colnames=["Predicted"],
        normalize="index",
    ) * 100

    # ========================================================
    # SPLIT BY REQUESTED FEEL
    # ========================================================
    tight_df = df[df["requested_feel"] == "tight"].copy()
    loose_df = df[df["requested_feel"] == "loose"].copy()
    swung_df = df[df["requested_feel"] == "swung"].copy()

    # ========================================================
    # FULL THRESHOLD SUCCESS
    #
    # IMPORTANT:
    #
    # Each row uses its OWN genre-specific swing threshold:
    #
    # funk / hiphop = 1.2
    # rock / jazz   = 1.5
    # ========================================================
    tight_threshold_success = (
        (tight_df["mean_swing_ratio"] < tight_df["swing_threshold"])
        & (tight_df["std_offset"] <= TIGHT_LOOSE_THRESHOLD)
    ).mean()

    loose_threshold_success = (
        (loose_df["mean_swing_ratio"] < loose_df["swing_threshold"])
        & (loose_df["std_offset"] > TIGHT_LOOSE_THRESHOLD)
    ).mean()

    swing_threshold_success = (
        swung_df["mean_swing_ratio"] >= swung_df["swing_threshold"]
    ).mean()

    # ========================================================
    # ACCURACY BY GENRE
    # ========================================================
    genre_accuracy = df.groupby("genre")["correct"].mean()

    # ========================================================
    # ACCURACY BY GENRE × REQUESTED FEEL
    # ========================================================
    genre_feel_accuracy = df.pivot_table(
        index="genre",
        columns="requested_feel",
        values="correct",
        aggfunc="mean",
    ) * 100

    genre_feel_accuracy = genre_feel_accuracy.reindex(
        columns=["tight", "loose", "swung"]
    )

    # ========================================================
    # HELPER FUNCTION
    # ========================================================
    def get_feature_mean(feel, feature):
        if feel in feature_means.index:
            return float(feature_means.loc[feel, feature])
        return np.nan

    # ========================================================
    # SUMMARY
    # ========================================================
    summary = {
        # Sample counts
        "N original test patterns": df["index"].nunique(),
        "N generated examples": len(df),

        # Thresholds
        "Funk/Hiphop swing threshold": FUNK_HIPHOP_SWING_RATIO_THRESHOLD,
        "Rock/Jazz swing threshold": SWING_RATIO_THRESHOLD,
        "Tight/Loose offset threshold": TIGHT_LOOSE_THRESHOLD,

        # Overall accuracy
        "Overall feel accuracy": overall_accuracy,

        # Accuracy per requested feel
        "Tight classification accuracy": class_accuracy.get("tight", np.nan),
        "Loose classification accuracy": class_accuracy.get("loose", np.nan),
        "Swung classification accuracy": class_accuracy.get("swung", np.nan),

        # Threshold success
        "Tight threshold success": tight_threshold_success,
        "Loose threshold success": loose_threshold_success,
        "Swung threshold success": swing_threshold_success,

        # TIGHT - BOTH FEATURES
        "Tight - mean std offset": get_feature_mean("tight", "std_offset"),
        "Tight - mean swing ratio": get_feature_mean("tight", "mean_swing_ratio"),

        # LOOSE - BOTH FEATURES
        "Loose - mean std offset": get_feature_mean("loose", "std_offset"),
        "Loose - mean swing ratio": get_feature_mean("loose", "mean_swing_ratio"),

        # SWUNG - BOTH FEATURES
        "Swung - mean std offset": get_feature_mean("swung", "std_offset"),
        "Swung - mean swing ratio": get_feature_mean("swung", "mean_swing_ratio"),
    }

    return (
        df,
        summary,
        confusion,
        confusion_percent,
        feature_means,
        feature_stds,
        genre_feel_accuracy,
        genre_feature_means,
    )


# ============================================================
# RUN EXPERIMENT 3
# ============================================================

(
    feel_df,
    experiment3_results,
    feel_confusion,
    feel_confusion_percent,
    feel_feature_means,
    feel_feature_stds,
    feel_genre_accuracy,
    feel_genre_feature_means,
) = evaluate_feel_control(
    model=model,
    dataset=test_dataset,
    device=device,
    n_examples=15000,
    seed=456,
)


# ============================================================
# MAIN RESULTS TABLE
# ============================================================

experiment3_table = pd.DataFrame({
    "Metric": experiment3_results.keys(),
    "Value": experiment3_results.values(),
})

print("\nEXPERIMENT 3 - FEEL CONTROL")
print("=" * 75)
display(experiment3_table.round(5))


# ============================================================
# MEAN SWING RATIO + STD OFFSET FOR ALL FEELS
# ============================================================

print("\nMEAN GROOVE FEATURES BY REQUESTED FEEL")
print("=" * 75)

mean_feature_table = feel_feature_means.rename(columns={
    "mean_swing_ratio": "Mean Swing Ratio",
    "std_offset": "Mean Std Offset",
})

display(mean_feature_table.round(5))


# ============================================================
# FEATURE VARIABILITY
# ============================================================

print("\nFEATURE VARIABILITY BY REQUESTED FEEL")
print("=" * 75)

std_feature_table = feel_feature_stds.rename(columns={
    "mean_swing_ratio": "Swing Ratio SD",
    "std_offset": "Std Offset SD",
})

display(std_feature_table.round(5))


# ============================================================
# COMBINED FEATURE TABLE
# ============================================================

combined_feature_table = pd.DataFrame(index=feel_feature_means.index)

combined_feature_table["Swing Ratio Mean"] = feel_feature_means["mean_swing_ratio"]
combined_feature_table["Swing Ratio SD"] = feel_feature_stds["mean_swing_ratio"]
combined_feature_table["Offset SD Mean"] = feel_feature_means["std_offset"]
combined_feature_table["Offset SD SD"] = feel_feature_stds["std_offset"]

print("\nCOMBINED FEATURE RESULTS")
print("=" * 75)
display(combined_feature_table.round(5))


# ============================================================
# ACCURACY BY GENRE × FEEL
#
# Important because swing threshold now depends on genre.
# ============================================================

print("\nFEEL ACCURACY (%) - GENRE × REQUESTED FEEL")
print("=" * 75)
display(feel_genre_accuracy.round(2))


# ============================================================
# FEATURE MEANS BY GENRE × FEEL
# ============================================================

print("\nMEAN FEATURES - GENRE × REQUESTED FEEL")
print("=" * 75)

display(
    feel_genre_feature_means.rename(columns={
        "mean_swing_ratio": "Mean Swing Ratio",
        "std_offset": "Mean Std Offset",
    }).round(5)
)


# ============================================================
# CONFUSION MATRIX - COUNTS
# ============================================================

print("\nFEEL CONFUSION MATRIX - COUNTS")
print("=" * 75)
display(feel_confusion)


# ============================================================
# CONFUSION MATRIX - ROW %
# ============================================================

print("\nFEEL CONFUSION MATRIX - ROW %")
print("=" * 75)
display(feel_confusion_percent.round(2))

In [ ]:
# ============================================================
# EXPERIMENT 4
# LATENT DIVERSITY
# ============================================================

@torch.inference_mode()
def evaluate_latent_diversity(
    model,
    dataset,
    device,
    n_examples=100,
    n_latent_samples=20,
    seed=789,
):
    model.eval()

    rng = np.random.default_rng(seed)

    n_examples = min(
        n_examples,
        len(dataset),
    )

    selected_indices = rng.choice(
        len(dataset),
        size=n_examples,
        replace=False,
    )

    torch.manual_seed(seed)

    records = []

    for idx in selected_indices:

        example = dataset[int(idx)]

        hit_np = example["hits"].numpy()

        active = hit_np > 0.5

        if active.sum() == 0:
            continue

        tempo_value = float(
            example["tempo"]
        )

        genre = str(
            example["raw_style"]
        )

        feel = str(
            example["feel"]
        )

        dynamics = str(
            example["overall_dynamics"]
        )

        prompt = create_attribute_prompt(
            genre=genre,
            feel=feel,
            overall_dynamics=dynamics,
            random_prompt=False,
        )

        generated_offsets = []
        generated_velocities = []

        # ----------------------------------------------------
        # Same prompt, same score, different z
        # ----------------------------------------------------

        for _ in range(n_latent_samples):

            (
                pred_offset_np,
                pred_velocity_np,
                _,
            ) = predict_with_prompt_vae(
                model=model,
                hit_matrix=hit_np,
                tempo_value=tempo_value,
                prompt_text=prompt,

                # None causes a fresh N(0, I) sample
                z=None,

                temperature=1.0,
            )

            generated_offsets.append(
                pred_offset_np
            )

            generated_velocities.append(
                pred_velocity_np
            )

        generated_offsets = np.stack(
            generated_offsets,
            axis=0,
        )

        generated_velocities = np.stack(
            generated_velocities,
            axis=0,
        )

        # Shapes:
        # [n_latent_samples, 32, 9]

        active_offsets = (
            generated_offsets[:, active]
        )

        active_velocities = (
            generated_velocities[:, active]
        )

        # ----------------------------------------------------
        # SD across z for each hit, then average over hits.
        # ----------------------------------------------------

        offset_diversity_normalized = (
            active_offsets
            .std(axis=0)
            .mean()
        )

        velocity_diversity_normalized = (
            active_velocities
            .std(axis=0)
            .mean()
        )

        # Convert into interpretable units.
        step_duration_ms = (
            60.0
            / tempo_value
            / SUBDIVISION_PER_BEAT
            * 1000.0
        )

        offset_diversity_ms = (
            offset_diversity_normalized
            * step_duration_ms
        )

        velocity_diversity_midi = (
            velocity_diversity_normalized
            * 127.0
        )

        # ----------------------------------------------------
        # Pairwise differences between generated performances
        # ----------------------------------------------------

        pairwise_offset_differences = []
        pairwise_velocity_differences = []

        for i in range(n_latent_samples):

            for j in range(
                i + 1,
                n_latent_samples,
            ):

                offset_difference = (
                    np.abs(
                        active_offsets[i]
                        - active_offsets[j]
                    )
                    .mean()
                )

                velocity_difference = (
                    np.abs(
                        active_velocities[i]
                        - active_velocities[j]
                    )
                    .mean()
                )

                pairwise_offset_differences.append(
                    offset_difference
                    * step_duration_ms
                )

                pairwise_velocity_differences.append(
                    velocity_difference
                    * 127.0
                )

        records.append(
            {
                "index": int(idx),
                "genre": genre,
                "feel": feel,
                "dynamics": dynamics,

                "offset_diversity_normalized":
                    offset_diversity_normalized,

                "offset_diversity_ms":
                    offset_diversity_ms,

                "velocity_diversity_normalized":
                    velocity_diversity_normalized,

                "velocity_diversity_midi":
                    velocity_diversity_midi,

                "mean_pairwise_offset_difference_ms":
                    np.mean(
                        pairwise_offset_differences
                    ),

                "mean_pairwise_velocity_difference_midi":
                    np.mean(
                        pairwise_velocity_differences
                    ),
            }
        )

    df = pd.DataFrame(records)

    summary = {
        "N test patterns":
            len(df),

        "Latent samples per pattern":
            n_latent_samples,

        "Mean offset diversity (ms)":
            df[
                "offset_diversity_ms"
            ].mean(),

        "Mean velocity diversity (MIDI)":
            df[
                "velocity_diversity_midi"
            ].mean(),

        "Mean pairwise timing difference (ms)":
            df[
                "mean_pairwise_offset_difference_ms"
            ].mean(),

        "Mean pairwise velocity difference (MIDI)":
            df[
                "mean_pairwise_velocity_difference_midi"
            ].mean(),
    }

    return df, summary


# ------------------------------------------------------------
# Run Experiment 4
# ------------------------------------------------------------

latent_df, experiment4_results = (
    evaluate_latent_diversity(
        model=model,
        dataset=test_dataset,
        device=device,

        # 100 patterns × 20 z samples
        n_examples=1500,
        n_latent_samples=20,

        seed=789,
    )
)

experiment4_table = pd.DataFrame(
    {
        "Metric": experiment4_results.keys(),
        "Value": experiment4_results.values(),
    }
)

print("\nEXPERIMENT 4 - LATENT DIVERSITY")
print("=" * 60)

display(experiment4_table.round(5))

In [ ]:
# ============================================================
# EXPERIMENT 5 - GENRE × FEEL × DYNAMICS INTERACTION MATRIX
#
# Full factorial design:
#
#   4 genres
#   × 3 feels
#   × 3 dynamics
#   = 36 conditions per test pattern
#
# SAME for all 36 conditions:
#   - hit pattern
#   - tempo
#   - latent z
#
# CHANGED:
#   - genre prompt
#   - feel prompt
#   - dynamics prompt
#
# Measurements:
#   - mean velocity
#   - std offset
#   - mean swing ratio
#
# Genre-dependent FEEL classification:
#
# Funk / Hiphop:
#   swung if swing_ratio >= 1.2
#
# Rock / Jazz:
#   swung if swing_ratio >= 1.5
#
# Otherwise:
#   tight if std_offset <= 0.1
#   loose if std_offset > 0.1
#
# Swing is given priority.
# ============================================================


# ============================================================
# THRESHOLDS
# ============================================================

SWING_RATIO_THRESHOLD = 1.5
FUNK_HIPHOP_SWING_RATIO_THRESHOLD = 1.2
TIGHT_LOOSE_THRESHOLD = 0.1


# ============================================================
# GENRE-SPECIFIC SWING THRESHOLD
# ============================================================

def get_swing_threshold(genre):
    """
    Return the swing threshold appropriate for the
    requested genre.

    Funk / hiphop -> 1.2
    Rock / jazz   -> 1.5
    """

    genre = str(genre).lower().strip()

    if genre in ["funk", "hiphop"]:
        return FUNK_HIPHOP_SWING_RATIO_THRESHOLD

    return SWING_RATIO_THRESHOLD


# ============================================================
# FEEL CLASSIFIER
# ============================================================

def classify_feel(
    mean_swing_ratio,
    std_offset,
    genre,
):
    """
    Classify generated performance using genre-dependent
    swing thresholds.

    Priority:
        1. swung
        2. tight
        3. loose
    """

    swing_threshold = get_swing_threshold(
        genre
    )

    if mean_swing_ratio >= swing_threshold:
        return "swung"

    elif std_offset <= TIGHT_LOOSE_THRESHOLD:
        return "tight"

    else:
        return "loose"


# ============================================================
# MAIN EVALUATION
# ============================================================

@torch.inference_mode()
def evaluate_condition_interactions(
    model,
    dataset,
    device,
    n_examples=200,
    seed=1001,
):

    model.eval()

    # --------------------------------------------------------
    # CONDITIONS
    # --------------------------------------------------------

    genres = [
        "funk",
        "rock",
        "jazz",
        "hiphop",
    ]

    feels = [
        "tight",
        "loose",
        "swung",
    ]

    dynamics_levels = [
        "soft",
        "medium",
        "loud",
    ]

    # --------------------------------------------------------
    # SAMPLE TEST PATTERNS
    # --------------------------------------------------------

    rng = np.random.default_rng(seed)

    n_examples = min(
        n_examples,
        len(dataset),
    )

    selected_indices = rng.choice(
        len(dataset),
        size=n_examples,
        replace=False,
    )

    torch.manual_seed(seed)

    records = []

    # ========================================================
    # LOOP THROUGH TEST PATTERNS
    # ========================================================

    for pattern_number, idx in enumerate(
        selected_indices
    ):

        example = dataset[int(idx)]

        hit_np = example["hits"].numpy()

        active = hit_np > 0.5

        if active.sum() < 2:
            continue

        tempo_value = float(
            example["tempo"]
        )

        original_genre = str(
            example["raw_style"]
        ).lower().strip()

        # ----------------------------------------------------
        # ONE latent vector for ALL 36 conditions.
        #
        # Differences therefore come from conditioning,
        # rather than different random z values.
        # ----------------------------------------------------

        shared_z = torch.randn(
            1,
            model.latent_dim,
            device=device,
        )

        # ====================================================
        # FULL 4 × 3 × 3 FACTORIAL LOOP
        # ====================================================

        for genre in genres:

            # Genre-specific swing threshold
            swing_threshold = get_swing_threshold(
                genre
            )

            for feel in feels:

                for dynamics in dynamics_levels:

                    # ----------------------------------------
                    # Create condition prompt
                    # ----------------------------------------

                    prompt = create_attribute_prompt(
                        genre=genre,
                        feel=feel,
                        overall_dynamics=dynamics,
                        random_prompt=False,
                    )

                    # ----------------------------------------
                    # Generate performance
                    # ----------------------------------------

                    (
                        pred_offset_np,
                        pred_velocity_np,
                        _,
                    ) = predict_with_prompt_vae(
                        model=model,
                        hit_matrix=hit_np,
                        tempo_value=tempo_value,
                        prompt_text=prompt,
                        z=shared_z,
                    )

                    # ----------------------------------------
                    # Extract groove features
                    # ----------------------------------------

                    (
                        mean_swing_ratio,
                        subdivision,
                        large_8th_note_swing,
                        mean_offset,
                        mean_velocity,
                        std_offset,
                        std_velocity,
                    ) = measure_groove_features(
                        hit_np.copy(),
                        pred_offset_np.copy(),
                        pred_velocity_np.copy(),
                    )

                    # ----------------------------------------
                    # Genre-aware feel classification
                    # ----------------------------------------

                    predicted_feel = classify_feel(
                        mean_swing_ratio=
                            mean_swing_ratio,

                        std_offset=
                            std_offset,

                        genre=
                            genre,
                    )

                    feel_correct = (
                        predicted_feel
                        == feel
                    )

                    # ----------------------------------------
                    # Save result
                    # ----------------------------------------

                    records.append(
                        {
                            "pattern_index":
                                int(idx),

                            "original_genre":
                                original_genre,

                            "prompt_genre":
                                genre,

                            "feel":
                                feel,

                            "dynamics":
                                dynamics,

                            "tempo":
                                tempo_value,

                            # --------------------------------
                            # Threshold actually used
                            # --------------------------------

                            "swing_threshold":
                                float(
                                    swing_threshold
                                ),

                            # --------------------------------
                            # Groove measurements
                            # --------------------------------

                            "mean_velocity":
                                float(
                                    mean_velocity
                                ),

                            "std_velocity":
                                float(
                                    std_velocity
                                ),

                            "std_offset":
                                float(
                                    std_offset
                                ),

                            "mean_swing_ratio":
                                float(
                                    mean_swing_ratio
                                ),

                            # --------------------------------
                            # Feel prediction
                            # --------------------------------

                            "predicted_feel":
                                predicted_feel,

                            "feel_correct":
                                bool(
                                    feel_correct
                                ),
                        }
                    )

    # ========================================================
    # RESULTS DATAFRAME
    # ========================================================

    df = pd.DataFrame(records)

    # ========================================================
    # AGGREGATE EVERY GENRE × FEEL × DYNAMICS CONDITION
    # ========================================================

    condition_summary = (
        df
        .groupby(
            [
                "prompt_genre",
                "feel",
                "dynamics",
            ]
        )
        .agg(
            mean_velocity=(
                "mean_velocity",
                "mean",
            ),

            velocity_sd=(
                "mean_velocity",
                "std",
            ),

            mean_std_offset=(
                "std_offset",
                "mean",
            ),

            std_offset_sd=(
                "std_offset",
                "std",
            ),

            mean_swing_ratio=(
                "mean_swing_ratio",
                "mean",
            ),

            swing_ratio_sd=(
                "mean_swing_ratio",
                "std",
            ),

            feel_accuracy=(
                "feel_correct",
                "mean",
            ),

            n=(
                "pattern_index",
                "count",
            ),
        )
        .reset_index()
    )

    # Convert accuracy to %
    condition_summary[
        "feel_accuracy"
    ] *= 100

    # ========================================================
    # MATRIX 1 - MEAN VELOCITY
    #
    # Rows = genre × feel
    # Columns = soft / medium / loud
    # ========================================================

    velocity_matrix = (
        condition_summary
        .pivot_table(
            index=[
                "prompt_genre",
                "feel",
            ],
            columns="dynamics",
            values="mean_velocity",
        )
    )

    velocity_matrix = velocity_matrix.reindex(
        columns=[
            "soft",
            "medium",
            "loud",
        ]
    )

    # ========================================================
    # MATRIX 2 - OFFSET STANDARD DEVIATION
    # ========================================================

    offset_matrix = (
        condition_summary
        .pivot_table(
            index=[
                "prompt_genre",
                "feel",
            ],
            columns="dynamics",
            values="mean_std_offset",
        )
    )

    offset_matrix = offset_matrix.reindex(
        columns=[
            "soft",
            "medium",
            "loud",
        ]
    )

    # ========================================================
    # MATRIX 3 - SWING RATIO
    # ========================================================

    swing_matrix = (
        condition_summary
        .pivot_table(
            index=[
                "prompt_genre",
                "feel",
            ],
            columns="dynamics",
            values="mean_swing_ratio",
        )
    )

    swing_matrix = swing_matrix.reindex(
        columns=[
            "soft",
            "medium",
            "loud",
        ]
    )

    # ========================================================
    # MATRIX 4 - FEEL ACCURACY
    #
    # This is where the genre-dependent swing threshold
    # directly affects the results.
    #
    # Rows = genre × requested feel
    # Columns = dynamics
    # ========================================================

    feel_accuracy_matrix = (
        df
        .pivot_table(
            index=[
                "prompt_genre",
                "feel",
            ],
            columns="dynamics",
            values="feel_correct",
            aggfunc="mean",
        )
        * 100
    )

    feel_accuracy_matrix = (
        feel_accuracy_matrix.reindex(
            columns=[
                "soft",
                "medium",
                "loud",
            ]
        )
    )

    # ========================================================
    # GENRE × FEEL MATRICES
    #
    # Average across dynamics
    # ========================================================

    genre_feel_velocity = (
        df
        .groupby(
            [
                "prompt_genre",
                "feel",
            ]
        )["mean_velocity"]
        .mean()
        .unstack("feel")
    )

    genre_feel_offset = (
        df
        .groupby(
            [
                "prompt_genre",
                "feel",
            ]
        )["std_offset"]
        .mean()
        .unstack("feel")
    )

    genre_feel_swing = (
        df
        .groupby(
            [
                "prompt_genre",
                "feel",
            ]
        )["mean_swing_ratio"]
        .mean()
        .unstack("feel")
    )

    genre_feel_accuracy = (
        df
        .pivot_table(
            index="prompt_genre",
            columns="feel",
            values="feel_correct",
            aggfunc="mean",
        )
        * 100
    )

    genre_feel_accuracy = (
        genre_feel_accuracy.reindex(
            columns=[
                "tight",
                "loose",
                "swung",
            ]
        )
    )

    # ========================================================
    # GENRE × DYNAMICS
    #
    # Average across feel
    # ========================================================

    genre_dynamics_velocity = (
        df
        .groupby(
            [
                "prompt_genre",
                "dynamics",
            ]
        )["mean_velocity"]
        .mean()
        .unstack("dynamics")
    )

    genre_dynamics_feel_accuracy = (
        df
        .pivot_table(
            index="prompt_genre",
            columns="dynamics",
            values="feel_correct",
            aggfunc="mean",
        )
        * 100
    )

    genre_dynamics_feel_accuracy = (
        genre_dynamics_feel_accuracy
        .reindex(
            columns=[
                "soft",
                "medium",
                "loud",
            ]
        )
    )

    # ========================================================
    # FEEL × DYNAMICS
    #
    # Average across genre
    # ========================================================

    feel_dynamics_velocity = (
        df
        .groupby(
            [
                "feel",
                "dynamics",
            ]
        )["mean_velocity"]
        .mean()
        .unstack("dynamics")
    )

    feel_dynamics_offset = (
        df
        .groupby(
            [
                "feel",
                "dynamics",
            ]
        )["std_offset"]
        .mean()
        .unstack("dynamics")
    )

    feel_dynamics_swing = (
        df
        .groupby(
            [
                "feel",
                "dynamics",
            ]
        )["mean_swing_ratio"]
        .mean()
        .unstack("dynamics")
    )

    feel_dynamics_accuracy = (
        df
        .pivot_table(
            index="feel",
            columns="dynamics",
            values="feel_correct",
            aggfunc="mean",
        )
        * 100
    )

    feel_dynamics_accuracy = (
        feel_dynamics_accuracy.reindex(
            index=[
                "tight",
                "loose",
                "swung",
            ],

            columns=[
                "soft",
                "medium",
                "loud",
            ],
        )
    )

    # ========================================================
    # FEEL CONFUSION MATRICES
    # ========================================================

    feel_confusion = pd.crosstab(
        df["feel"],
        df["predicted_feel"],
        rownames=[
            "Requested"
        ],
        colnames=[
            "Predicted"
        ],
        normalize=False,
    )

    feel_confusion_percent = (
        pd.crosstab(
            df["feel"],
            df["predicted_feel"],
            rownames=[
                "Requested"
            ],
            colnames=[
                "Predicted"
            ],
            normalize="index",
        )
        * 100
    )

    # ========================================================
    # OVERALL SUMMARY
    # ========================================================

    summary = {
        "N test patterns":
            df[
                "pattern_index"
            ].nunique(),

        "N generated performances":
            len(df),

        "Overall feel accuracy (%)":
            df[
                "feel_correct"
            ].mean()
            * 100,

        "Funk/Hiphop swing threshold":
            FUNK_HIPHOP_SWING_RATIO_THRESHOLD,

        "Rock/Jazz swing threshold":
            SWING_RATIO_THRESHOLD,

        "Tight/Loose offset threshold":
            TIGHT_LOOSE_THRESHOLD,
    }

    # ========================================================
    # RETURN EVERYTHING
    # ========================================================

    return {
        "raw":
            df,

        "summary":
            summary,

        "condition_summary":
            condition_summary,

        "velocity_matrix":
            velocity_matrix,

        "offset_matrix":
            offset_matrix,

        "swing_matrix":
            swing_matrix,

        "feel_accuracy_matrix":
            feel_accuracy_matrix,

        "genre_feel_velocity":
            genre_feel_velocity,

        "genre_feel_offset":
            genre_feel_offset,

        "genre_feel_swing":
            genre_feel_swing,

        "genre_feel_accuracy":
            genre_feel_accuracy,

        "genre_dynamics_velocity":
            genre_dynamics_velocity,

        "genre_dynamics_feel_accuracy":
            genre_dynamics_feel_accuracy,

        "feel_dynamics_velocity":
            feel_dynamics_velocity,

        "feel_dynamics_offset":
            feel_dynamics_offset,

        "feel_dynamics_swing":
            feel_dynamics_swing,

        "feel_dynamics_accuracy":
            feel_dynamics_accuracy,

        "feel_confusion":
            feel_confusion,

        "feel_confusion_percent":
            feel_confusion_percent,
    }


# ============================================================
# RUN EXPERIMENT
#
# 1500 patterns × 36 conditions
# = 54,000 generated performances
# ============================================================

interaction_results = evaluate_condition_interactions(
    model=model,
    dataset=test_dataset,
    device=device,
    n_examples=1500,
    seed=1001,
)


# ============================================================
# BASIC INFORMATION
# ============================================================

interaction_df = interaction_results[
    "raw"
]

print(
    "\nGENRE × FEEL × DYNAMICS INTERACTION EXPERIMENT"
)

print("=" * 80)

print(
    "Test patterns:",
    interaction_df[
        "pattern_index"
    ].nunique()
)

print(
    "Generated performances:",
    len(interaction_df)
)

print(
    "Conditions per pattern:",
    4 * 3 * 3
)


# ============================================================
# SUMMARY
# ============================================================

print("\nSUMMARY")
print("=" * 80)

summary_table = pd.DataFrame(
    {
        "Metric":
            interaction_results[
                "summary"
            ].keys(),

        "Value":
            interaction_results[
                "summary"
            ].values(),
    }
)

display(
    summary_table.round(4)
)


# ============================================================
# FULL 36-CONDITION SUMMARY
# ============================================================

print("\nFULL 36-CONDITION SUMMARY")
print("=" * 80)

display(
    interaction_results[
        "condition_summary"
    ].round(5)
)


# ============================================================
# MATRIX 1 - VELOCITY
# ============================================================

print("\nMEAN VELOCITY")
print(
    "Rows = Genre × Feel | "
    "Columns = Dynamics"
)
print("=" * 80)

display(
    interaction_results[
        "velocity_matrix"
    ].round(5)
)


# ============================================================
# MATRIX 2 - OFFSET SD
# ============================================================

print("\nMEAN OFFSET STANDARD DEVIATION")
print(
    "Rows = Genre × Feel | "
    "Columns = Dynamics"
)
print("=" * 80)

display(
    interaction_results[
        "offset_matrix"
    ].round(5)
)


# ============================================================
# MATRIX 3 - SWING RATIO
# ============================================================

print("\nMEAN SWING RATIO")
print(
    "Rows = Genre × Feel | "
    "Columns = Dynamics"
)
print("=" * 80)

display(
    interaction_results[
        "swing_matrix"
    ].round(5)
)


# ============================================================
# MATRIX 4 - FEEL ACCURACY
#
# THIS MATRIX USES:
#
# Funk/Hiphop swing threshold = 1.2
# Rock/Jazz swing threshold   = 1.5
# ============================================================

print("\nFEEL ACCURACY (%)")
print(
    "Rows = Genre × Requested Feel | "
    "Columns = Dynamics"
)
print("=" * 80)

display(
    interaction_results[
        "feel_accuracy_matrix"
    ].round(2)
)


# ============================================================
# GENRE × FEEL
# ============================================================

print("\nGENRE × FEEL - MEAN VELOCITY")
print("=" * 80)

display(
    interaction_results[
        "genre_feel_velocity"
    ].round(5)
)


print("\nGENRE × FEEL - OFFSET SD")
print("=" * 80)

display(
    interaction_results[
        "genre_feel_offset"
    ].round(5)
)


print("\nGENRE × FEEL - SWING RATIO")
print("=" * 80)

display(
    interaction_results[
        "genre_feel_swing"
    ].round(5)
)


print("\nGENRE × FEEL - FEEL ACCURACY (%)")
print("=" * 80)

display(
    interaction_results[
        "genre_feel_accuracy"
    ].round(2)
)


# ============================================================
# GENRE × DYNAMICS
# ============================================================

print("\nGENRE × DYNAMICS - MEAN VELOCITY")
print("=" * 80)

display(
    interaction_results[
        "genre_dynamics_velocity"
    ].round(5)
)


print(
    "\nGENRE × DYNAMICS - "
    "FEEL ACCURACY (%)"
)
print("=" * 80)

display(
    interaction_results[
        "genre_dynamics_feel_accuracy"
    ].round(2)
)


# ============================================================
# FEEL × DYNAMICS
# ============================================================

print("\nFEEL × DYNAMICS - MEAN VELOCITY")
print("=" * 80)

display(
    interaction_results[
        "feel_dynamics_velocity"
    ].round(5)
)


print("\nFEEL × DYNAMICS - OFFSET SD")
print("=" * 80)

display(
    interaction_results[
        "feel_dynamics_offset"
    ].round(5)
)


print("\nFEEL × DYNAMICS - SWING RATIO")
print("=" * 80)

display(
    interaction_results[
        "feel_dynamics_swing"
    ].round(5)
)


print("\nFEEL × DYNAMICS - FEEL ACCURACY (%)")
print("=" * 80)

display(
    interaction_results[
        "feel_dynamics_accuracy"
    ].round(2)
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

print("\nFEEL CONFUSION MATRIX - COUNTS")
print("=" * 80)

display(
    interaction_results[
        "feel_confusion"
    ]
)


print("\nFEEL CONFUSION MATRIX - ROW %")
print("=" * 80)

display(
    interaction_results[
        "feel_confusion_percent"
    ].round(2)
)

In [ ]:
# ============================================================
# EXPERIMENT 5 - GENRE × FEEL × DYNAMICS ACCURACY MATRIX
#
# Tests interaction between:
#
#   4 genres × 3 feels × 3 dynamics
#   = 36 prompt conditions per test pattern
#
# FEEL CLASSIFICATION:
#
# swung:
#   swing_ratio >= 1.5
#
# tight:
#   swing_ratio < 1.5
#   AND std_offset <= 0.1
#
# loose:
#   swing_ratio < 1.5
#   AND std_offset > 0.1
#
#
# DYNAMICS CLASSIFICATION:
#
# soft:
#   mean_velocity <= 0.4832396
#
# medium:
#   0.4832396 < mean_velocity < 0.6237068
#
# loud:
#   mean_velocity >= 0.6237068
#
#
# JOINT ACCURACY:
#
#   correct feel
#   AND
#   correct dynamics
#
# Genre is used as a conditioning variable.
# Genre itself is NOT classified in this experiment.
# ============================================================


# ============================================================
# THRESHOLDS
# ============================================================

SWING_RATIO_THRESHOLD = 1.5
TIGHT_LOOSE_THRESHOLD = 0.1

SOFT_THRESHOLD = 0.4832396
LOUD_THRESHOLD = 0.6237068


# ============================================================
# FEEL CLASSIFIER
# ============================================================

def classify_feel(
    mean_swing_ratio,
    std_offset,
):
    """
    Classify generated performance as:
        tight
        loose
        swung
    """

    # Swing gets priority
    if mean_swing_ratio >= SWING_RATIO_THRESHOLD:
        return "swung"

    elif std_offset <= TIGHT_LOOSE_THRESHOLD:
        return "tight"

    else:
        return "loose"


# ============================================================
# DYNAMICS CLASSIFIER
# ============================================================

def classify_dynamics(
    mean_velocity,
):
    """
    Classify generated performance as:
        soft
        medium
        loud

    using fixed velocity thresholds.
    """

    if mean_velocity <= SOFT_THRESHOLD:
        return "soft"

    elif mean_velocity >= LOUD_THRESHOLD:
        return "loud"

    else:
        return "medium"


# ============================================================
# MAIN EVALUATION FUNCTION
# ============================================================

@torch.inference_mode()
def evaluate_interaction_accuracy(
    model,
    dataset,
    device,
    n_examples=500,
    seed=1001,
):

    model.eval()

    # --------------------------------------------------------
    # Prompt conditions
    # --------------------------------------------------------

    genres = [
        "funk",
        "rock",
        "jazz",
        "hiphop",
    ]

    feels = [
        "tight",
        "loose",
        "swung",
    ]

    dynamics_levels = [
        "soft",
        "medium",
        "loud",
    ]

    # --------------------------------------------------------
    # Select test patterns
    # --------------------------------------------------------

    rng = np.random.default_rng(seed)

    n_examples = min(
        n_examples,
        len(dataset),
    )

    selected_indices = rng.choice(
        len(dataset),
        size=n_examples,
        replace=False,
    )

    torch.manual_seed(seed)

    records = []

    # ========================================================
    # GENERATION LOOP
    # ========================================================

    for idx in selected_indices:

        example = dataset[int(idx)]

        hit_np = example["hits"].numpy()

        active = hit_np > 0.5

        if active.sum() < 2:
            continue

        tempo_value = float(
            example["tempo"]
        )

        original_genre = str(
            example["raw_style"]
        )

        # ----------------------------------------------------
        # ONE shared z for ALL 36 prompt conditions.
        #
        # This means changes are due to:
        #
        #   genre
        #   feel
        #   dynamics
        #
        # rather than random latent differences.
        # ----------------------------------------------------

        shared_z = torch.randn(
            1,
            model.latent_dim,
            device=device,
        )

        # ====================================================
        # FULL FACTORIAL:
        #
        # 4 genres
        # × 3 feels
        # × 3 dynamics
        # = 36 conditions
        # ====================================================

        for prompt_genre in genres:

            for requested_feel in feels:

                for requested_dynamics in dynamics_levels:

                    # ----------------------------------------
                    # Build prompt
                    # ----------------------------------------

                    prompt = create_attribute_prompt(
                        genre=prompt_genre,
                        feel=requested_feel,
                        overall_dynamics=requested_dynamics,
                        random_prompt=False,
                    )

                    # ----------------------------------------
                    # Generate performance
                    # ----------------------------------------

                    (
                        pred_offset_np,
                        pred_velocity_np,
                        _,
                    ) = predict_with_prompt_vae(
                        model=model,
                        hit_matrix=hit_np,
                        tempo_value=tempo_value,
                        prompt_text=prompt,
                        z=shared_z,
                    )

                    # ----------------------------------------
                    # Extract groove features
                    # ----------------------------------------

                    (
                        mean_swing_ratio,
                        subdivision,
                        large_8th_note_swing,
                        mean_offset,
                        mean_velocity,
                        std_offset,
                        std_velocity,
                    ) = measure_groove_features(
                        hit_np.copy(),
                        pred_offset_np.copy(),
                        pred_velocity_np.copy(),
                    )

                    # =================================================
                    # CLASSIFY FEEL
                    # =================================================

                    predicted_feel = classify_feel(
                        mean_swing_ratio=
                            mean_swing_ratio,

                        std_offset=
                            std_offset,
                    )

                    feel_correct = (
                        predicted_feel
                        == requested_feel
                    )

                    # =================================================
                    # CLASSIFY DYNAMICS
                    # =================================================

                    predicted_dynamics = classify_dynamics(
                        mean_velocity
                    )

                    dynamics_correct = (
                        predicted_dynamics
                        == requested_dynamics
                    )

                    # =================================================
                    # JOINT ACCURACY
                    # =================================================

                    joint_correct = (
                        feel_correct
                        and dynamics_correct
                    )

                    # =================================================
                    # STORE RESULT
                    # =================================================

                    records.append(
                        {
                            "pattern_index":
                                int(idx),

                            "original_genre":
                                original_genre,

                            "prompt_genre":
                                prompt_genre,

                            "requested_feel":
                                requested_feel,

                            "requested_dynamics":
                                requested_dynamics,

                            "tempo":
                                tempo_value,

                            # -----------------------------
                            # Groove measurements
                            # -----------------------------

                            "mean_velocity":
                                float(mean_velocity),

                            "std_offset":
                                float(std_offset),

                            "mean_swing_ratio":
                                float(mean_swing_ratio),

                            # -----------------------------
                            # Predictions
                            # -----------------------------

                            "predicted_feel":
                                predicted_feel,

                            "predicted_dynamics":
                                predicted_dynamics,

                            # -----------------------------
                            # Accuracy
                            # -----------------------------

                            "feel_correct":
                                bool(feel_correct),

                            "dynamics_correct":
                                bool(dynamics_correct),

                            "joint_correct":
                                bool(joint_correct),
                        }
                    )

    # ========================================================
    # RAW DATAFRAME
    # ========================================================

    df = pd.DataFrame(records)

    # ========================================================
    # OVERALL ACCURACY
    # ========================================================

    overall_feel_accuracy = (
        df["feel_correct"].mean()
        * 100
    )

    overall_dynamics_accuracy = (
        df["dynamics_correct"].mean()
        * 100
    )

    overall_joint_accuracy = (
        df["joint_correct"].mean()
        * 100
    )

    # ========================================================
    # MATRIX 1
    #
    # FEEL ACCURACY
    #
    # Rows:
    #   Genre × Requested Feel
    #
    # Columns:
    #   Requested Dynamics
    #
    # ========================================================

    feel_accuracy_full = (
        df
        .pivot_table(
            index=[
                "prompt_genre",
                "requested_feel",
            ],
            columns="requested_dynamics",
            values="feel_correct",
            aggfunc="mean",
        )
        * 100
    )

    feel_accuracy_full = (
        feel_accuracy_full.reindex(
            columns=[
                "soft",
                "medium",
                "loud",
            ]
        )
    )

    # ========================================================
    # MATRIX 2
    #
    # DYNAMICS ACCURACY
    #
    # Rows:
    #   Genre × Requested Feel
    #
    # Columns:
    #   Requested Dynamics
    #
    # ========================================================

    dynamics_accuracy_full = (
        df
        .pivot_table(
            index=[
                "prompt_genre",
                "requested_feel",
            ],
            columns="requested_dynamics",
            values="dynamics_correct",
            aggfunc="mean",
        )
        * 100
    )

    dynamics_accuracy_full = (
        dynamics_accuracy_full.reindex(
            columns=[
                "soft",
                "medium",
                "loud",
            ]
        )
    )

    # ========================================================
    # MATRIX 3
    #
    # JOINT ACCURACY
    #
    # BOTH:
    #
    #   requested feel correct
    #   requested dynamics correct
    #
    # ========================================================

    joint_accuracy_full = (
        df
        .pivot_table(
            index=[
                "prompt_genre",
                "requested_feel",
            ],
            columns="requested_dynamics",
            values="joint_correct",
            aggfunc="mean",
        )
        * 100
    )

    joint_accuracy_full = (
        joint_accuracy_full.reindex(
            columns=[
                "soft",
                "medium",
                "loud",
            ]
        )
    )

    # ========================================================
    # FEEL ACCURACY - GENRE × FEEL
    #
    # Dynamics averaged.
    # ========================================================

    feel_accuracy_genre_feel = (
        df
        .pivot_table(
            index="prompt_genre",
            columns="requested_feel",
            values="feel_correct",
            aggfunc="mean",
        )
        * 100
    )

    feel_accuracy_genre_feel = (
        feel_accuracy_genre_feel.reindex(
            columns=[
                "tight",
                "loose",
                "swung",
            ]
        )
    )

    # ========================================================
    # DYNAMICS ACCURACY - GENRE × FEEL
    #
    # Soft / medium / loud averaged.
    # ========================================================

    dynamics_accuracy_genre_feel = (
        df
        .pivot_table(
            index="prompt_genre",
            columns="requested_feel",
            values="dynamics_correct",
            aggfunc="mean",
        )
        * 100
    )

    dynamics_accuracy_genre_feel = (
        dynamics_accuracy_genre_feel.reindex(
            columns=[
                "tight",
                "loose",
                "swung",
            ]
        )
    )

    # ========================================================
    # JOINT ACCURACY - GENRE × FEEL
    #
    # Dynamics averaged.
    # ========================================================

    joint_accuracy_genre_feel = (
        df
        .pivot_table(
            index="prompt_genre",
            columns="requested_feel",
            values="joint_correct",
            aggfunc="mean",
        )
        * 100
    )

    joint_accuracy_genre_feel = (
        joint_accuracy_genre_feel.reindex(
            columns=[
                "tight",
                "loose",
                "swung",
            ]
        )
    )

    # ========================================================
    # DYNAMICS ACCURACY - GENRE × DYNAMICS
    #
    # Feel averaged.
    # ========================================================

    dynamics_accuracy_genre_dynamics = (
        df
        .pivot_table(
            index="prompt_genre",
            columns="requested_dynamics",
            values="dynamics_correct",
            aggfunc="mean",
        )
        * 100
    )

    dynamics_accuracy_genre_dynamics = (
        dynamics_accuracy_genre_dynamics.reindex(
            columns=[
                "soft",
                "medium",
                "loud",
            ]
        )
    )

    # ========================================================
    # FEEL ACCURACY - FEEL × DYNAMICS
    #
    # Genre averaged.
    # ========================================================

    feel_accuracy_feel_dynamics = (
        df
        .pivot_table(
            index="requested_feel",
            columns="requested_dynamics",
            values="feel_correct",
            aggfunc="mean",
        )
        * 100
    )

    feel_accuracy_feel_dynamics = (
        feel_accuracy_feel_dynamics.reindex(
            index=[
                "tight",
                "loose",
                "swung",
            ],
            columns=[
                "soft",
                "medium",
                "loud",
            ],
        )
    )

    # ========================================================
    # DYNAMICS ACCURACY - FEEL × DYNAMICS
    #
    # Genre averaged.
    # ========================================================

    dynamics_accuracy_feel_dynamics = (
        df
        .pivot_table(
            index="requested_feel",
            columns="requested_dynamics",
            values="dynamics_correct",
            aggfunc="mean",
        )
        * 100
    )

    dynamics_accuracy_feel_dynamics = (
        dynamics_accuracy_feel_dynamics.reindex(
            index=[
                "tight",
                "loose",
                "swung",
            ],
            columns=[
                "soft",
                "medium",
                "loud",
            ],
        )
    )

    # ========================================================
    # JOINT ACCURACY - FEEL × DYNAMICS
    # ========================================================

    joint_accuracy_feel_dynamics = (
        df
        .pivot_table(
            index="requested_feel",
            columns="requested_dynamics",
            values="joint_correct",
            aggfunc="mean",
        )
        * 100
    )

    joint_accuracy_feel_dynamics = (
        joint_accuracy_feel_dynamics.reindex(
            index=[
                "tight",
                "loose",
                "swung",
            ],
            columns=[
                "soft",
                "medium",
                "loud",
            ],
        )
    )

    # ========================================================
    # DYNAMICS CONFUSION MATRIX
    # ========================================================

    dynamics_confusion = pd.crosstab(
        df["requested_dynamics"],
        df["predicted_dynamics"],
        rownames=["Requested"],
        colnames=["Predicted"],
        normalize=False,
    )

    dynamics_confusion_percent = (
        pd.crosstab(
            df["requested_dynamics"],
            df["predicted_dynamics"],
            rownames=["Requested"],
            colnames=["Predicted"],
            normalize="index",
        )
        * 100
    )

    # ========================================================
    # FEEL CONFUSION MATRIX
    # ========================================================

    feel_confusion = pd.crosstab(
        df["requested_feel"],
        df["predicted_feel"],
        rownames=["Requested"],
        colnames=["Predicted"],
        normalize=False,
    )

    feel_confusion_percent = (
        pd.crosstab(
            df["requested_feel"],
            df["predicted_feel"],
            rownames=["Requested"],
            colnames=["Predicted"],
            normalize="index",
        )
        * 100
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    summary = {

        "N test patterns":
            df["pattern_index"].nunique(),

        "N generated performances":
            len(df),

        "Feel accuracy (%)":
            overall_feel_accuracy,

        "Dynamics accuracy (%)":
            overall_dynamics_accuracy,

        "Joint feel + dynamics accuracy (%)":
            overall_joint_accuracy,

        "Soft threshold":
            SOFT_THRESHOLD,

        "Loud threshold":
            LOUD_THRESHOLD,

        "Swing ratio threshold":
            SWING_RATIO_THRESHOLD,

        "Tight/loose threshold":
            TIGHT_LOOSE_THRESHOLD,
    }

    return {

        "raw":
            df,

        "summary":
            summary,

        "feel_accuracy_full":
            feel_accuracy_full,

        "dynamics_accuracy_full":
            dynamics_accuracy_full,

        "joint_accuracy_full":
            joint_accuracy_full,

        "feel_accuracy_genre_feel":
            feel_accuracy_genre_feel,

        "dynamics_accuracy_genre_feel":
            dynamics_accuracy_genre_feel,

        "joint_accuracy_genre_feel":
            joint_accuracy_genre_feel,

        "dynamics_accuracy_genre_dynamics":
            dynamics_accuracy_genre_dynamics,

        "feel_accuracy_feel_dynamics":
            feel_accuracy_feel_dynamics,

        "dynamics_accuracy_feel_dynamics":
            dynamics_accuracy_feel_dynamics,

        "joint_accuracy_feel_dynamics":
            joint_accuracy_feel_dynamics,

        "dynamics_confusion":
            dynamics_confusion,

        "dynamics_confusion_percent":
            dynamics_confusion_percent,

        "feel_confusion":
            feel_confusion,

        "feel_confusion_percent":
            feel_confusion_percent,
    }


# ============================================================
# RUN EXPERIMENT
#
# 500 test patterns × 36 conditions
# = 18,000 generated performances
#
# Change to 15000 if you want to use the entire test set.
# ============================================================

accuracy_interactions = evaluate_interaction_accuracy(
    model=model,
    dataset=test_dataset,
    device=device,
    n_examples=1500,
    seed=1001,
)


# ============================================================
# SUMMARY
# ============================================================

summary_table = pd.DataFrame(
    {
        "Metric":
            accuracy_interactions[
                "summary"
            ].keys(),

        "Value":
            accuracy_interactions[
                "summary"
            ].values(),
    }
)

print("\nGENRE × FEEL × DYNAMICS ACCURACY EXPERIMENT")
print("=" * 80)

display(
    summary_table.round(4)
)


# ============================================================
# FULL FEEL ACCURACY MATRIX
# ============================================================

print("\nFEEL ACCURACY (%)")

display(
    accuracy_interactions[
        "feel_accuracy_full"
    ].round(2)
)


# ============================================================
# FULL DYNAMICS ACCURACY MATRIX
# ============================================================

print("\nDYNAMICS ACCURACY (%)")

display(
    accuracy_interactions[
        "dynamics_accuracy_full"
    ].round(2)
)


# ============================================================
# FULL JOINT ACCURACY MATRIX
#
# This is probably the main table for the paper.
#
# Example:
#
# rock + swung + loud = 85%
#
# means 85% of generated performances were BOTH:
#
#   classified as swung
#   AND
#   classified as loud
# ============================================================

print("\nJOINT FEEL + DYNAMICS ACCURACY (%)")
print("Rows = Genre × Requested Feel")
print("Columns = Requested Dynamics")
print("=" * 80)

display(
    accuracy_interactions[
        "joint_accuracy_full"
    ].round(2)
)


# ============================================================
# GENRE × FEEL ACCURACY
# ============================================================

print("\nFEEL ACCURACY (%) - GENRE × FEEL")
print("=" * 80)

display(
    accuracy_interactions[
        "feel_accuracy_genre_feel"
    ].round(2)
)


print("\nDYNAMICS ACCURACY (%) - GENRE × FEEL")
print("=" * 80)

display(
    accuracy_interactions[
        "dynamics_accuracy_genre_feel"
    ].round(2)
)


print("\nJOINT ACCURACY (%) - GENRE × FEEL")
print("=" * 80)

display(
    accuracy_interactions[
        "joint_accuracy_genre_feel"
    ].round(2)
)


# ============================================================
# GENRE × DYNAMICS
# ============================================================

print("\nDYNAMICS ACCURACY (%) - GENRE × DYNAMICS")
print("=" * 80)

display(
    accuracy_interactions[
        "dynamics_accuracy_genre_dynamics"
    ].round(2)
)


# ============================================================
# FEEL × DYNAMICS
# ============================================================

print("\nFEEL ACCURACY (%) - FEEL × DYNAMICS")
print("=" * 80)

display(
    accuracy_interactions[
        "feel_accuracy_feel_dynamics"
    ].round(2)
)


print("\nDYNAMICS ACCURACY (%) - FEEL × DYNAMICS")
print("=" * 80)

display(
    accuracy_interactions[
        "dynamics_accuracy_feel_dynamics"
    ].round(2)
)


print("\nJOINT ACCURACY (%) - FEEL × DYNAMICS")
print("=" * 80)

display(
    accuracy_interactions[
        "joint_accuracy_feel_dynamics"
    ].round(2)
)


# ============================================================
# DYNAMICS CONFUSION MATRIX
# ============================================================

print("\nDYNAMICS CONFUSION MATRIX - COUNTS")
print("=" * 80)

display(
    accuracy_interactions[
        "dynamics_confusion"
    ]
)


print("\nDYNAMICS CONFUSION MATRIX - ROW %")
print("=" * 80)

display(
    accuracy_interactions[
        "dynamics_confusion_percent"
    ].round(2)
)


# ============================================================
# FEEL CONFUSION MATRIX
# ============================================================

print("\nFEEL CONFUSION MATRIX - COUNTS")
print("=" * 80)

display(
    accuracy_interactions[
        "feel_confusion"
    ]
)


print("\nFEEL CONFUSION MATRIX - ROW %")
print("=" * 80)

display(
    accuracy_interactions[
        "feel_confusion_percent"
    ].round(2)
)